<a href="https://colab.research.google.com/github/minoak/AIFFEL_quest_eng/blob/main/LLM_Application/LLM02/Day2_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

last modified date : 2026.05  
제작 : 모두의연구소

# Day 2 실습 — Advanced·Modular RAG + RAGAS 평가

# 들어가며

Day 1에서는 가장 기본형인 **Naive RAG** 파이프라인을 직접 구현해 보았습니다. 이번 실습에서는 한국어 QA 벤치마크 **KorQuAD v1** 데이터셋 위에서 **Advanced·Modular RAG** 의 핵심 기법(Multi-Query, RAG-Fusion, HyDE, Reranking, Self-RAG)을 단계적으로 적용하고, 그 결과를 **RAGAS** 로 정량 평가합니다.

이번 실습이 끝나면 다음을 직접 말할 수 있게 됩니다.
- Naive RAG 대비 **어떤 단계**를 보강하면 정답률이 올라가는가
- Multi-Query / RAG-Fusion / HyDE / Reranker / Self-RAG 는 각각 **어떤 코드 라인**으로 적용하는가
- RAGAS 의 4대 지표(Faithfulness · Answer Relevance · Context Precision · Context Recall)는 어떻게 계산되고 어떻게 읽는가
- 내 RAG 가 ‘얼마나 좋아졌는지’를 **숫자로** 보여주는 방법

## Step 0 : 설치와 준비  
Day 1과 동일하게 Colab에서 진행한다고 가정합니다.

In [1]:
# Colab pre-installed langchain 0.3 / ragas 0.1~0.4 를 ragas 0.2.10 호환 조합으로 정리합니다.
# 처음 실행 시 약 3~5분 걸립니다. 진행률 출력을 보면서 기다리세요 (멈춘 게 아닙니다).

# 1) 기존 langchain / ragas 패키지 제거 — 버전 충돌로 인한 pip resolver 백트래킹 방지
!pip uninstall -y ragas ragas-experimental langchain langchain-core langchain-community langchain-openai langchain-text-splitters langchain-chroma

# 2) 0.2 시리즈 패치 버전까지 핀 설치 — resolver 부담 최소화 (-q 제거해서 진행률 보이게)
!pip install --no-cache-dir \
    "ragas==0.2.10" \
    "langchain==0.2.17" \
    "langchain-core==0.2.43" \
    "langchain-community==0.2.19" \
    "langchain-openai==0.1.25" \
    "langchain-text-splitters==0.2.4" \
    "langchain-chroma==0.1.4" \
    pypdf chromadb tiktoken sentence-transformers datasets nest_asyncio pandas

Found existing installation: ragas 0.2.10
Uninstalling ragas-0.2.10:
  Successfully uninstalled ragas-0.2.10
Found existing installation: langchain 0.2.17
Uninstalling langchain-0.2.17:
  Successfully uninstalled langchain-0.2.17
Found existing installation: langchain-core 0.2.43
Uninstalling langchain-core-0.2.43:
  Successfully uninstalled langchain-core-0.2.43
Found existing installation: langchain-community 0.2.19
Uninstalling langchain-community-0.2.19:
  Successfully uninstalled langchain-community-0.2.19
Found existing installation: langchain-openai 0.1.25
Uninstalling langchain-openai-0.1.25:
  Successfully uninstalled langchain-openai-0.1.25
Found existing installation: langchain-text-splitters 0.2.4
Uninstalling langchain-text-splitters-0.2.4:
  Successfully uninstalled langchain-text-splitters-0.2.4
Found existing installation: langchain-chroma 0.1.4
Uninstalling langchain-chroma-0.1.4:
  Successfully uninstalled langchain-chroma-0.1.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import os
# chromadb 익명 통계 전송 끄기 — posthog SDK 인자 충돌로 ERROR 로그가 뜨는 것 방지
os.environ["ANONYMIZED_TELEMETRY"] = "False"

import nest_asyncio
nest_asyncio.apply()  # RAGAS가 Colab의 비동기 이벤트 루프와 충돌하지 않도록

In [6]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('아이펠')

## Step 1 : KorQuAD v1 위에서 Naive RAG 베이스라인 만들기

Day 1에서 만든 RAG 파이프라인을 한국어 QA 벤치마크 **KorQuAD v1** 위에 다시 한 번 올립니다. 이후 단계는 모두 이 베이스라인 위에 ‘덧붙이는’ 방식입니다.

- HuggingFace `datasets` 로 KorQuAD v1 자동 다운로드 (별도 PDF 업로드 불필요)
- 일부만 샘플링해 토큰 비용 통제 (unique context 약 200개)
- Embedding → VectorStore → Retriever → LLM
- 검색 전략은 단순 `similarity` (top-k)

**📥 데이터셋**: <https://huggingface.co/datasets/KorQuAD/squad_kor_v1>

In [7]:
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import tiktoken, random

tokenizer = tiktoken.get_encoding("cl100k_base")
def tiktoken_len(text):
    return len(tokenizer.encode(text))

# 1) 데이터셋 로드 + 2000개 샘플링 + context 중복 제거 → unique 약 800개
#    (Vector DB 가 크면 Reranker 의 정밀도 개선 효과가 더 또렷하게 보입니다.
#     인덱싱 토큰 비용 약 0.01 USD 추가)
raw_ds = load_dataset("squad_kor_v1", split="validation").shuffle(seed=42).select(range(2000))

unique = {}
for ex in raw_ds:
    if ex["context"] not in unique:
        unique[ex["context"]] = ex["title"]
context_docs = [Document(page_content=c, metadata={"title": t}) for c, t in unique.items()]

# 2) chunk 단위로 분할 (KorQuAD context는 짧지만 길이 균질화를 위해 splitter 사용)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function=tiktoken_len)
docs = splitter.split_documents(context_docs)

# 3) Embedding & Chroma 적재 — chunk 약 800개를 한 번에 넣으면 chromadb 의 batch limit
#    (Colab 환경에서 보통 5461) 또는 OpenAI rate limit 에 걸릴 수 있어
#    100개씩 배치로 add_documents 합니다.
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma(embedding_function=embedding)
BATCH = 100
for i in range(0, len(docs), BATCH):
    db.add_documents(docs[i:i+BATCH])

# 4) Retriever (Naive: similarity)
naive_retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

# 5) LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print(f"베이스라인 준비 완료 — unique context: {len(context_docs)}, chunks: {len(docs)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

squad_kor_v1/train-00000-of-00001.parque(…):   0%|          | 0.00/11.6M [00:00<?, ?B/s]

squad_kor_v1/validation-00000-of-00001.p(…):   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60407 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5774 [00:00<?, ? examples/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


베이스라인 준비 완료 — unique context: 847, chunks: 1264


베이스라인 RAG로 간단한 질의를 던져 답이 나오는지 확인합니다.

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT = ChatPromptTemplate.from_template(
    "다음 문서를 참고해 질문에 한국어로 간결하게 답하세요. 문서에 없는 내용은 만들지 마세요.\n\n"
    "[문서]\n{context}\n\n"
    "[질문]\n{question}\n\n"
    "[답변]"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

naive_chain = (
    {"context": naive_retriever | format_docs,
     "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# 데이터셋에서 첫 질문 하나를 뽑아 테스트
TEST_Q = raw_ds[0]["question"]
print("Q:", TEST_Q)
print("A:", naive_chain.invoke(TEST_Q))

Q: 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


A: 대중교통체계입니다.


## Step 2 : Pre-retrieval 강화 — Multi-Query Retrieval  

사용자가 던진 질문 하나로만 검색하면 ‘다른 표현’으로 적힌 정답을 놓칠 수 있습니다. **Multi-Query Retrieval**은 LLM에게 ‘같은 의도의 다른 질문 N개’를 만들게 시킨 뒤, 각 질문으로 병렬 검색하고 결과를 합칩니다.

LangChain은 이를 한 클래스로 제공합니다.

In [9]:
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
)

# 어떤 ‘유사 질문’으로 확장되는지 로그로 확인 가능
docs_mq = multi_query_retriever.invoke(TEST_Q)
print(f"검색된 문서 수: {len(docs_mq)}")
print("---")
print(docs_mq[0].page_content[:300])

INFO:langchain.retrievers.multi_query:Generated queries: ['2004년 이명박 서울시장이 재직할 때 어떤 주요 개선 사항이 있었나요?  ', '이명박이 2004년 서울시장으로서 추진한 주요 정책이나 변화는 무엇인가요?  ', '2004년 이명박 서울시장 재임 기간 동안 어떤 프로젝트나 계획이 크게 발전했나요?']


검색된 문서 수: 6
---
2010년 한나라당 당내경선에서 나경원, 김충환 등의 경쟁자를 물리치고, 민선 5기 지방선거에서 서울시장 재선에 도전했다. 6월 2일에 치뤄진 지방선거에서 개표 초반에 한명숙 후보에게 뒤지다가, 후반 강남 3구의 개표가 시작되면서 역전하여 민선 5기 제34대 서울특별시장으로 재선되었다. 구체적으로 강남구(+59,206, +25.68%), 서초구(+43,820, +23.66%), 송파구(+23,814, +8.19%), 강동구(+11,097, +5.33%), 용산구(+8,579, +8.24%), 양천구(+1,078, +0.51%), 영


[링크 텍스트](https://)## Step 2.5 : RAG-Fusion — Multi-Query + RRF로 묶어내기

Day2_1 노트에서 “꼭 짚고 가라”고 했던 패턴 중 하나가 **RAG-Fusion** 입니다. Step 2의 Multi-Query는 ‘유사 질문 N개로 병렬 검색’ 까지만 했는데, **RAG-Fusion** 은 그 N개 검색 결과를 **Reciprocal Rank Fusion (RRF)** 라는 간단한 공식으로 합쳐 ‘여러 쿼리에서 공통으로 상위에 떴던 문서’ 를 최상단으로 끌어올립니다.

RRF 점수 공식:

$$
\text{score}(d) = \sum_{i=1}^{N} \frac{1}{k + \text{rank}_i(d)}
$$

- $\text{rank}_i(d)$ : i번째 쿼리의 결과에서 문서 $d$ 의 순위 (1부터)
- $k$ : 스무딩 상수 (관례적으로 60)

아래 셀에서는 (1) sub-query 생성, (2) 각 sub-query 로 검색, (3) **RRF 함수는 여러분이 직접 채우기**, (4) 결과 확인까지 한 번에 해봅니다.

In [15]:
from collections import defaultdict

# (1) sub-query 생성 — Multi-Query 가 내부적으로 하는 일을 명시적으로 노출 (한국어)
SUBQUERY_PROMPT = ChatPromptTemplate.from_template(
    "당신은 검색 보조 AI 입니다. 다음 질문과 의미는 같지만 표현이 다른 4개의 한국어 검색 쿼리를 만드세요. "
    "오직 4개의 쿼리만 한 줄에 하나씩 출력하고, 번호나 다른 설명은 붙이지 마세요.\n\n질문: {question}"
)

def fan_out_queries(question, n=4):
    raw = (SUBQUERY_PROMPT | llm | StrOutputParser()).invoke({"question": question})
    return [q.strip() for q in raw.split("\n") if q.strip()][:n]


# (2) RRF 함수 — TODO: 여러분이 직접 채워보세요
def reciprocal_rank_fusion(results_per_query, k=60, top_k=3):
    scores = defaultdict(float)
    docs_by_key = {}

    # TODO 1: 점수 누적
    for docs in results_per_query:
        for rank, doc in enumerate(docs):
            key = doc.page_content
            scores[key] += 1.0 / (k + rank + 1)  # rank는 0부터니까 +1
            docs_by_key[key] = doc

    # TODO 2: 정렬 + 상위 top_k 반환
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [docs_by_key[key] for key, _ in ranked[:top_k]]
    scores = defaultdict(float)
    docs_by_key = {}

    # TODO 1: 각 쿼리의 결과 리스트를 순회하면서 문서마다 RRF 점수를 누적해 보세요.
    #   힌트:
    #     for docs in results_per_query:
    #         for rank, doc in enumerate(docs):  # rank 는 0부터
    #             key = doc.page_content
    #             scores[key] += 1.0 / (k + rank + 1)
    #             docs_by_key[key] = doc


    # TODO 2: scores 값이 큰 순서로 정렬해서 상위 top_k 개의 Document 를 반환하세요.
    #   힌트:
    #     ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    #     return [docs_by_key[k] for k, _ in ranked[:top_k]]

    return []


# (3) 한 번 돌려보기
sub_queries = fan_out_queries(TEST_Q)
print(f"확장 질문 {len(sub_queries)}개:")
for q in sub_queries:
    print(" -", q)

results_per_q = [db.similarity_search(q, k=5) for q in sub_queries]
fused = reciprocal_rank_fusion(results_per_q, k=60, top_k=3)

print("\nRAG-Fusion top-1 문서:")
print(fused[0].page_content[:300] if fused else "(아직 TODO 가 비어 있어 결과가 없습니다)")

확장 질문 4개:
 - 2004년 이명박 서울시장 재직 중 개선한 사항은?
 - 이명박이 2004년 서울시장으로서 개선한 내용은 무엇인가?
 - 2004년 서울시장 이명박이 전면적으로 개선한 것은 어떤 것인가?
 - 이명박 서울시장 시절 2004년에 개선된 것은 무엇인가?

RAG-Fusion top-1 문서:
2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도 했다. 하지만 새 교통체계가 정착되면서 많은 긍정적인 효과를 가져오게 된다. 중앙버스차로 도입으로 버스의 평균 속도가 증가하여 정시에 도착하는 빈도가 늘어났고 환승제도로 인한 교


## Step 3 : 패턴 ② HyDE — 가상의 ‘정답’으로 진짜 정답 찾기  

질문은 짧은 의문문, 정답은 긴 평서문이라 둘의 임베딩이 의외로 멀 수 있습니다. **HyDE(Hypothetical Document Embeddings)** 는 검색 전에 LLM에게 ‘가상의 정답’을 쓰게 한 뒤, 그 가상 답변을 임베딩해서 검색합니다.

직접 구현해 보겠습니다.

In [11]:
HYDE_PROMPT = ChatPromptTemplate.from_template(
    "당신은 해당 분야 전문가입니다. 다음 질문에 대해 그럴듯한 한국어 답변 한 문단을 작성하세요. "
    "확실하지 않다면 가장 합리적인 추측을 적어주세요.\n\n"
    "질문: {question}\n\n가상 답변:"
)

hyde_generator = HYDE_PROMPT | llm | StrOutputParser()

def hyde_retrieve(question, k=3):
    """질문 → 가상의 답변 → 가상 답변을 임베딩해 검색"""
    hypothetical = hyde_generator.invoke({"question": question})
    return db.similarity_search(hypothetical, k=k), hypothetical

docs_hyde, hyp = hyde_retrieve(TEST_Q)
print("가상 답변(HyDE):\n", hyp[:300], "\n---")
print("검색된 문서 수:", len(docs_hyde))
print("첫 문서:", docs_hyde[0].page_content[:200])

가상 답변(HyDE):
 2004년 이명박이 서울시장으로 재직하던 시절, 그는 서울의 교통 체계를 전면적으로 개선하는 데 주력했습니다. 특히, 그는 '서울시 교통체계 개선 계획'을 통해 대중교통의 효율성을 높이고, 도로 혼잡을 줄이기 위한 다양한 정책을 시행했습니다. 이 과정에서 지하철 노선 확장과 버스 전용 차선 도입, 그리고 자전거 도로의 확충 등이 이루어졌습니다. 이러한 노력은 서울시민의 이동 편의를 증대시키고, 대기 오염 문제를 완화하는 데 기여했습니다. 
---
검색된 문서 수: 3
첫 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 4 : Post-retrieval 강화 — Cross-Encoder Reranking (multilingual)

검색 결과를 그대로 LLM 에 넘기지 않고, **Cross-encoder reranker** 가 (질문, 문단)을 함께 보면서 진짜 관련도를 다시 점수화합니다. 정밀도가 15~30% 개선되는 게 일반적인 보고입니다.

한국어 문서를 다루고 있으므로 다국어를 지원하는 cross-encoder 를 사용합니다. `BAAI/bge-reranker-v2-m3` 는 한국어를 포함한 100개 이상 언어에서 동작합니다. 처음 실행 시 모델 다운로드(~2GB)가 발생합니다.

In [12]:
from sentence_transformers import CrossEncoder

# 다국어 cross-encoder (한국어 포함)
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rerank(query, docs, top_k=3):
    """검색된 docs 를 cross-encoder 로 다시 점수화해 상위 top_k 만 반환"""
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:top_k]]

candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(TEST_Q)
top3 = rerank(TEST_Q, candidates, top_k=3)
print(f"후보 {len(candidates)}개 → Reranker 로 상위 3개 선별")
print("최상위 문서:", top3[0].page_content[:200])

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

후보 10개 → Reranker 로 상위 3개 선별
최상위 문서: 2004년 서울시장 재직시절 대중교통체계를 전면적으로 개선하였다. 거리비례제를 도입하여 교통수단에 관계없이 이동한 거리에 비례해서 요금을 지불하게 바뀌면서 환승으로 인한 추가적인 교통비 부담이 없어졌다. 그 외에도 서울시 버스를 4종류로 나누고 버스 전용차로를 도로 중앙으로 옮기는 등의 많은 변화가 일시에 일어나면서 초기엔 시행착오로 인한 큰 불편을 겪기도


## Step 5 : Advanced RAG 체인 조립  

위에서 만든 컴포넌트들을 하나의 체인으로 묶습니다. **‘넓게 검색 → Reranker로 좁히기 → LLM 답변’** 패턴이 가장 흔히 쓰입니다.

In [17]:
def advanced_rag(question):
    # 1) 후보를 넓게 검색 (k=10)
    candidates = db.as_retriever(search_kwargs={"k": 10}).invoke(question)
    # 2) Cross-encoder 로 진짜 관련도 재정렬 후 상위 3개
    top = rerank(question, candidates, top_k=3)
    # 3) 프롬프트에 컨텍스트로 주입 → 답변
    context = format_docs(top)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": question})
    return answer, top

ans_adv, ctx_adv = advanced_rag(TEST_Q)
print("Advanced RAG 답변:\n", ans_adv)

Advanced RAG 답변:
 대중교통체계입니다.


## Step 5.5 : Self-RAG — 검색 필요성 판단 + 답변 자가 비평

Day2_1 노트에서 강조한 또 하나의 핵심 패턴, **Self-RAG** 입니다. Self-RAG의 핵심은 **LLM이 검색·답변 과정에 스스로 비평(critique)을 끼워 넣는다**는 점입니다.

이번 셀에서는 공식 Self-RAG 모델을 따로 받지 않고, **세 개의 작은 LLM 프롬프트**로 같은 흐름을 흉내내 봅니다.

1. **Retrieve 결정** — 질문이 들어오면, 외부 검색이 필요한지 LLM이 먼저 판단합니다. (`YES`/`NO` 한 단어)
2. **답변 생성** — `YES` 면 일반 RAG, `NO` 면 검색 없이 LLM 단독 답변.
3. **답변 자가 비평** — 생성된 답변이 컨텍스트에 충분히 근거하는지 LLM이 점검합니다. (`SUPPORTED` / `NOT_SUPPORTED`)
4. **보완 재시도** — `NOT_SUPPORTED` 면 Step 3의 **HyDE** 로 검색 쿼리를 바꿔 한 번 더 시도합니다.

코드 골격은 제공해 두었고, **두 군데 핵심 프롬프트만 여러분이 직접 채워주세요.**

In [19]:
# Self-RAG : retrieve 판단 + 자가 비평 + HyDE 재시도

# (1) 검색 필요성 판단 프롬프트 — TODO 1
RETRIEVE_DECISION_PROMPT = ChatPromptTemplate.from_template(
    # TODO: 사용자 질문이 외부 문서 검색이 필요하면 YES,
    #       일반 상식·계산·정의 등 LLM 자체 지식으로 답할 수 있으면 NO
    #       오직 한 단어(YES 또는 NO)만 출력하도록 한국어 프롬프트를 채우세요.
    "당신은 질문 분류 보조 AI 입니다. "
    "다음 질문에 답하려면 외부 문서 검색이 필요한지 판단하세요. "
    "외부 정보(고유명사, 사실, 사건, 통계)가 필요하면 YES, "
    "일반 상식이나 정의·계산만으로 답할 수 있으면 NO 입니다. "
    "오직 YES 또는 NO 한 단어만 출력하세요.\n\n"
    "질문: {question}"
)

# (2) 답변 자가 비평 프롬프트 — TODO 2
CRITIQUE_PROMPT = ChatPromptTemplate.from_template(
    # TODO: [문서] {context} 와 [답변] {answer} 가 주어졌을 때,
    #       답변이 문서 내용으로 충분히 뒷받침되면 SUPPORTED,
    #       아니면 NOT_SUPPORTED — 한 단어만 출력하도록 한국어 프롬프트를 채우세요.
  "당신은 답변 검증 보조 AI 입니다. "
    "주어진 [문서]만을 근거로 [답변]이 사실로 뒷받침되는지 판단하세요. "
    "문서에서 직접 확인 가능하면 SUPPORTED, "
    "문서에 없는 정보가 포함되어 있거나 확실하지 않으면 NOT_SUPPORTED 입니다. "
    "오직 SUPPORTED 또는 NOT_SUPPORTED 한 단어만 출력하세요.\n\n"
    "[문서]\n{context}\n\n"
    "[답변]\n{answer}"
)


def self_rag(question, max_retries=1, verbose=True):
    decision = (RETRIEVE_DECISION_PROMPT | llm | StrOutputParser()).invoke(
        {"question": question}).strip().upper()
    if verbose:
        print(f"[1] Retrieve 필요? -> {decision}")

    if decision.startswith("NO"):
        ans = llm.invoke(question).content
        if verbose:
            print("[2] LLM 단독 답변 사용")
        return ans, []

    docs = db.as_retriever(search_kwargs={"k": 3}).invoke(question)

    for attempt in range(max_retries + 1):
        answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "question": question})
        critique = (CRITIQUE_PROMPT | llm | StrOutputParser()).invoke(
            {"context": format_docs(docs), "answer": answer}).strip().upper()
        if verbose:
            print(f"[3] 시도 {attempt+1} — 자가 비평: {critique}")

        if "NOT" not in critique:
            return answer, docs

        if attempt < max_retries:
            hyp = hyde_generator.invoke({"question": question})
            docs = db.similarity_search(hyp, k=3)
            if verbose:
                print("[4] NOT_SUPPORTED -> HyDE 가상 답변으로 재검색")

    return answer, docs


ans_sr, ctx_sr = self_rag(TEST_Q)
print("\n=== Self-RAG 최종 답변 ===")
print(ans_sr)

[1] Retrieve 필요? -> YES
[3] 시도 1 — 자가 비평: NOT_SUPPORTED
[4] NOT_SUPPORTED -> HyDE 가상 답변으로 재검색
[3] 시도 2 — 자가 비평: NOT_SUPPORTED

=== Self-RAG 최종 답변 ===
대중교통체계입니다.


## Step 6 : RAGAS 평가용 데이터셋 만들기

RAGAS 는 네 가지 자료가 필요합니다.
- `user_input` — 사용자 질문
- `response`   — RAG 가 생성한 답변
- `retrieved_contexts` — RAG 가 참고한 문서들
- `reference`  — 모범 답안 (Ground Truth)

**KorQuAD 는 사람이 작성한 정답이 데이터셋에 이미 포함**되어 있어, `reference` 를 따로 작성할 필요 없이 그대로 가져다 씁니다. 같은 질문 셋을 **Naive RAG** 와 **Advanced RAG** 두 가지로 풀고 결과를 비교합니다.

토큰 비용 통제를 위해 평가 질문은 5개만 사용합니다. (늘리려면 `EVAL_N` 변경)

In [20]:
# 평가용 질문/정답 자동 추출 (KorQuAD)
EVAL_N = 20  # 평가 질문 수. 표본 분산을 줄이려 20개로 설정. 줄이려면 5~10.
eval_samples = list(raw_ds)[:EVAL_N]
questions = [ex["question"] for ex in eval_samples]
ground_truths = [ex["answers"]["text"][0] for ex in eval_samples]

# Naive RAG 로 답변 + 컨텍스트 수집
naive_answers, naive_contexts = [], []
for q in questions:
    ctx = naive_retriever.invoke(q)
    a = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": format_docs(ctx), "question": q})
    naive_answers.append(a)
    naive_contexts.append([d.page_content for d in ctx])

# Advanced RAG 로 답변 + 컨텍스트 수집
adv_answers, adv_contexts = [], []
for q in questions:
    a, ctx = advanced_rag(q)
    adv_answers.append(a)
    adv_contexts.append([d.page_content for d in ctx])

print(f"데이터셋 준비 완료 — {EVAL_N}개 질문 × 2개 파이프라인")

데이터셋 준비 완료 — 20개 질문 × 2개 파이프라인


In [22]:
from datasets import Dataset

def make_dataset(answers, contexts):
    return Dataset.from_dict({
        "user_input":         questions,
        "response":           answers,
        "retrieved_contexts": contexts,
        "reference":          ground_truths,
    })

naive_ds = make_dataset(naive_answers, naive_contexts)
adv_ds   = make_dataset(adv_answers,   adv_contexts)

## Step 7 : RAGAS로 4대 지표 계산하기  

Judge LLM은 `gpt-4o-mini`로, 임베딩은 `text-embedding-3-small`로 설정합니다.  
(Judge에 더 강한 모델을 쓰면 채점은 더 정교해지지만 비용이 늘어납니다.)

In [23]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness, answer_relevancy,
    context_precision, context_recall,
)

judge_llm  = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_emb  = OpenAIEmbeddings(model="text-embedding-3-small")
metrics    = [faithfulness, answer_relevancy,
              context_precision, context_recall]

print("=== Naive RAG 채점 ===")
naive_result = evaluate(naive_ds, metrics=metrics,
                        llm=judge_llm, embeddings=judge_emb,
                        raise_exceptions=False)

print("=== Advanced RAG 채점 ===")
adv_result = evaluate(adv_ds, metrics=metrics,
                      llm=judge_llm, embeddings=judge_emb,
                      raise_exceptions=False)

=== Naive RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== Advanced RAG 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

In [28]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

naive_df = naive_result.to_pandas()
adv_df   = adv_result.to_pandas()

def summary(df, label):
    cols = ["faithfulness", "answer_relevancy",
            "context_precision", "context_recall"]
    avg = df[cols].mean()
    avg.name = label
    return avg

compare = pd.concat([summary(naive_df, "Naive RAG"),
                     summary(adv_df,   "Advanced RAG")], axis=1)
print(compare.round(3))
print("\nDelta (Advanced - Naive):")
print((compare["Advanced RAG"] - compare["Naive RAG"]).round(3))

                   Naive RAG  Advanced RAG
faithfulness           0.650         0.875
answer_relevancy       0.304         0.272
context_precision      0.700         0.900
context_recall         0.750         0.900

Delta (Advanced - Naive):
faithfulness         0.225
answer_relevancy    -0.032
context_precision    0.200
context_recall       0.150
dtype: float64


### 결과 해석 가이드

위 비교표를 처음 보면 **‘Advanced 가 더 나쁜 거 아닌가?’** 라는 착각을 하기 쉽습니다. KorQuAD 위에서의 결과 해석 방법을 정리합니다.

**1. `context_precision` 의 개선 (+) 이 Advanced RAG 의 핵심 효과**
검색 결과의 ‘상단’에 정답 문단을 두는 일을 Reranker 가 잘 했다는 의미. Δ가 0.05~0.15 정도면 잘 작동.

**2. `context_recall = 1.0` 으로 포화될 수 있다**
unique context 가 800개 정도면 Naive top-3 에도 정답이 거의 항상 들어옵니다. 이 지표는 더 큰 DB(수만 문서)에서 차이가 드러납니다.

**3. `faithfulness` 가 살짝 떨어질 수 있다**
Reranker 가 컨텍스트를 ‘짧고 집중’ 시키면 LLM이 그 좁은 정보에서 답을 만들 때 일부 주장이 “미뒷받침” 으로 채점되어 점수가 약간 내려갈 수 있음. **정상 범위 (-0.1 이내)**.

**4. `answer_relevancy` 가 0.2~0.4 로 낮은 이유 — KorQuAD 의 구조적 특성**
KorQuAD 정답은 *‘대중교통체계’* 같이 한 단어~한 구절. RAG 답변도 짧게 나오는데, RAGAS 의 `answer_relevancy` 는 **답변에서 질문을 역추론**해 원래 질문과의 유사도를 계산합니다. 답변이 한 단어면 역추론이 흐려져 점수가 낮아집니다. **모델 잘못이 아닌 데이터셋 특성**.

**5. 표본 20개로도 Δ가 ±0.05 이내면 ‘차이 없음’으로 봐야 한다**
20문항에서 ±0.05 는 표본 noise. 더 확실한 판단이 필요하면 `scipy.stats.ttest_rel` 로 통계 검정을 하거나 50~100문항으로 늘려야 합니다.

**6. 한국어 짧은 정답 벤치마크의 한계**
KorQuAD/KLUE-MRC 처럼 정답이 짧은 extractive QA 벤치마크는 `context_precision` 위주로 평가 효과를 봐야 하고, `answer_relevancy` 는 절대값보다 **Naive 대비 상대 변화**로 읽어야 합니다.

### Quiz  
위 표에서 Advanced RAG가 가장 크게 개선한 지표는 무엇인가요? 그리고 그 지표는 우리가 적용한 **어떤 기법**과 가장 직접적으로 연결될까요?  

**Answer**:
  
faithfulness 가 가장 많이 올랐고 그 다음이 context_precision가 많이 올았음
둘 다 Reranker가 직접/간접적으로 만든 결과. Reranker가 더 정확한 문서를 상위로 올리면서, LLM이 받는 컨텍스트 품질이 좋아져 환각이 줄어들었다


## Step 8 : (선택) 평가 데이터를 LLM으로 자동 생성하기  

현업에서는 모범 답안(`reference`)을 사람이 직접 작성하는 게 가장 큰 부담입니다.  
RAGAS는 **원본 문서만 주면 Question·Reference·Context 한 세트를 자동으로 만들어 주는** 기능을 제공합니다.  
자세한 사용법은 공식 문서를 참고하세요.

https://docs.ragas.io/en/stable/getstarted/rag_testset_generation/

---
# 추가 실습 — KLUE-MRC 한국어 뉴스 MRC 벤치마크로 RAG 평가하기

메인 실습은 위키 기반 **KorQuAD v1** 으로 진행했습니다. 이번 추가 실습은 도메인을 바꿔, **한국어 뉴스 기사 기반의 MRC 벤치마크 KLUE-MRC** 위에서 같은 파이프라인을 처음부터 다시 조립해 봅니다.

**KLUE-MRC**
- 카카오·네이버 등 한국 NLP 팀이 함께 만든 한국어 표준 벤치마크 KLUE 의 MRC 태스크
- 한국어 **뉴스 기사** 기반 (KorQuAD 의 위키와 도메인이 다름)
- 사람이 직접 작성한 정답 포함
- **`is_impossible=True`** 인 답할 수 없는 질문도 일부 포함 → 데이터 필터링이 필요한 도전적 케이스

위키 기반 KorQuAD 와 비교했을 때 어떤 차이(질문 스타일, 검색 난이도, 점수 분포)가 나는지 직접 관찰해 보세요.

이번에도 일부만 샘플링해서 토큰 비용을 통제합니다.
- Vector DB 에 들어갈 unique context: 약 200개
- 평가 질문: 20개
- 예상 비용: GPT-4o-mini 기준 RAGAS 평가까지 합쳐서 약 \$0.10 ~ \$0.20

**📥 데이터셋 다운로드 / 출처**
- HuggingFace `datasets` 자동 다운로드: <https://huggingface.co/datasets/klue>
- KLUE 공식 사이트: <https://klue-benchmark.com/>
- KLUE 논문: <https://arxiv.org/abs/2105.09680>

> 다른 데이터셋으로 한 번 더 해보고 싶다면:  
> - MIRACL 한국어: <https://huggingface.co/datasets/miracl/miracl> (config: `ko`)  
> - 영어 SQuAD: <https://huggingface.co/datasets/rajpurkar/squad>

### Step A. 데이터셋 로드

`datasets` 라이브러리로 KLUE-MRC 를 한 줄에 받아옵니다. KLUE 는 여러 sub-task 가 있는 멀티태스크 벤치마크라서 config 이름 `"mrc"` 를 명시해야 합니다.

데이터셋 페이지: <https://huggingface.co/datasets/klue>

In [25]:
from datasets import load_dataset

ds_klue = load_dataset("klue", "mrc", split="validation")
print(ds_klue)
print("\n--- 샘플 1건 ---")
print({k: ds_klue[0][k] for k in ds_klue.column_names})

README.md: 0.00B [00:00, ?B/s]

mrc/train-00000-of-00001.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

mrc/validation-00000-of-00001.parquet:   0%|          | 0.00/8.68M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17554 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5841 [00:00<?, ? examples/s]

Dataset({
    features: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers'],
    num_rows: 5841
})

--- 샘플 1건 ---
{'title': 'BMW 코리아, 창립 25주년 기념 ‘BMW 코리아 25주년 에디션’ 한정 출시', 'context': 'BMW 코리아(대표 한상윤)는 창립 25주년을 기념하는 ‘BMW 코리아 25주년 에디션’을 한정 출시한다고 밝혔다. 이번 BMW 코리아 25주년 에디션(이하 25주년 에디션)은 BMW 3시리즈와 5시리즈, 7시리즈, 8시리즈 총 4종, 6개 모델로 출시되며, BMW 클래식 모델들로 선보인 바 있는 헤리티지 컬러가 차체에 적용돼 레트로한 느낌과 신구의 조화가 어우러진 차별화된 매력을 자랑한다. 먼저 뉴 320i 및 뉴 320d 25주년 에디션은 트림에 따라 옥스포드 그린(50대 한정) 또는 마카오 블루(50대 한정) 컬러가 적용된다. 럭셔리 라인에 적용되는 옥스포드 그린은 지난 1999년 3세대 3시리즈를 통해 처음 선보인 색상으로 짙은 녹색과 풍부한 펄이 오묘한 조화를 이루는 것이 특징이다. M 스포츠 패키지 트림에 적용되는 마카오 블루는 1988년 2세대 3시리즈를 통해 처음 선보인 바 있으며, 보랏빛 감도는 컬러감이 매력이다. 뉴 520d 25주년 에디션(25대 한정)은 프로즌 브릴리언트 화이트 컬러로 출시된다. BMW가 2011년에 처음 선보인 프로즌 브릴리언트 화이트는 한층 더 환하고 깊은 색감을 자랑하며, 특히 표면을 무광으로 마감해 특별함을 더했다. 뉴 530i 25주년 에디션(25대 한정)은 뉴 3시리즈 25주년 에디션에도 적용된 마카오 블루 컬러가 조합된다. 뉴 740Li 25주년 에디션(7대 한정)에는 말라카이트 그린 다크 색상이 적용된다. 잔잔하면서도 오묘한 깊은 녹색을 발산하는 말라카이트 그린 다크는 장식재로 

In [27]:
print("컬럼:", ds_klue.column_names)
print("샘플 1건:")
import json
print(json.dumps(ds_klue[0], ensure_ascii=False, indent=2)[:1500])

컬럼: ['title', 'context', 'news_category', 'source', 'guid', 'is_impossible', 'question_type', 'question', 'answers']
샘플 1건:
{
  "title": "BMW 코리아, 창립 25주년 기념 ‘BMW 코리아 25주년 에디션’ 한정 출시",
  "context": "BMW 코리아(대표 한상윤)는 창립 25주년을 기념하는 ‘BMW 코리아 25주년 에디션’을 한정 출시한다고 밝혔다. 이번 BMW 코리아 25주년 에디션(이하 25주년 에디션)은 BMW 3시리즈와 5시리즈, 7시리즈, 8시리즈 총 4종, 6개 모델로 출시되며, BMW 클래식 모델들로 선보인 바 있는 헤리티지 컬러가 차체에 적용돼 레트로한 느낌과 신구의 조화가 어우러진 차별화된 매력을 자랑한다. 먼저 뉴 320i 및 뉴 320d 25주년 에디션은 트림에 따라 옥스포드 그린(50대 한정) 또는 마카오 블루(50대 한정) 컬러가 적용된다. 럭셔리 라인에 적용되는 옥스포드 그린은 지난 1999년 3세대 3시리즈를 통해 처음 선보인 색상으로 짙은 녹색과 풍부한 펄이 오묘한 조화를 이루는 것이 특징이다. M 스포츠 패키지 트림에 적용되는 마카오 블루는 1988년 2세대 3시리즈를 통해 처음 선보인 바 있으며, 보랏빛 감도는 컬러감이 매력이다. 뉴 520d 25주년 에디션(25대 한정)은 프로즌 브릴리언트 화이트 컬러로 출시된다. BMW가 2011년에 처음 선보인 프로즌 브릴리언트 화이트는 한층 더 환하고 깊은 색감을 자랑하며, 특히 표면을 무광으로 마감해 특별함을 더했다. 뉴 530i 25주년 에디션(25대 한정)은 뉴 3시리즈 25주년 에디션에도 적용된 마카오 블루 컬러가 조합된다. 뉴 740Li 25주년 에디션(7대 한정)에는 말라카이트 그린 다크 색상이 적용된다. 잔잔하면서도 오묘한 깊은 녹색을 발산하는 말라카이트 그린 다크는 장식재로 활용되는 광물 말라카이트에서 유래됐다. 뉴 840i xDrive 그란쿠페 25주년 

### Step B. Context 추출 + 중복 제거 (+ is_impossible 필터링)

KLUE-MRC 에는 KorQuAD 에는 없는 **`is_impossible=True`** 케이스가 섞여 있습니다 (= context 만 보고는 답할 수 없는 질문). 평가용 ground_truth 가 비어 있으면 RAGAS 의 `context_recall` 이 깨지므로, 답이 있는 샘플만 남기세요.

- `ds_klue.filter(lambda x: not x["is_impossible"])` 로 답 있는 것만 추리고
- `shuffle(seed=42).select(range(300))` 으로 300개 샘플링
- 그 중 `context` 필드 기준으로 중복 제거 (보통 150~200개)
- 각각을 `Document(page_content=..., metadata={"title": ex["title"]})` 로 감싸 `context_docs` 에 담기

In [30]:
# TODO: 안내에 따라 context_docs 리스트를 만들어 보세요.
# 마지막에 len(context_docs) 와 context_docs[0].page_content[:200] 을 출력해서 확인.

import random
from langchain_core.documents import Document


# 1) is_impossible=False만 남기기 (답 있는 거)
ds_filtered = ds_klue.filter(lambda x: not x["is_impossible"])

# 2) 300개 샘플링
ds_sampled = ds_filtered.shuffle(seed=42).select(range(300))

# 3) context 기준 중복 제거
unique = {}
for ex in ds_sampled:
    if ex["context"] not in unique:
        unique[ex["context"]] = ex["title"]

# 4) Document 객체로 감싸기
from langchain_core.documents import Document
context_docs = [
    Document(page_content=c, metadata={"title": t})
    for c, t in unique.items()
]

print(f"unique context: {len(context_docs)}")
print(context_docs[0].page_content[:200])


Filter:   0%|          | 0/5841 [00:00<?, ? examples/s]

unique context: 299
국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


### Step C. Embedding + VectorStore

메인 실습에서 만든 `embedding` (`OpenAIEmbeddings(model="text-embedding-3-small")`) 을 그대로 재사용해, `context_docs` 로 새 Chroma DB `db_klue` 를 만드세요. (메인 실습의 `db` 변수를 덮어쓰지 마세요. 비교가 안 됩니다.)

> ⚠️ **batch 적재 필수** — KLUE-MRC 의 뉴스 context 는 평균 토큰 수가 커서, 150개 이상을 한 번에 `Chroma.from_documents` 로 넘기면 OpenAI embeddings 의 **300k 토큰/요청 한도** 에 걸려 `BadRequestError` 가 납니다. 메인 cell 8 처럼 100개씩 batch 로 `add_documents` 호출하세요:
> ```python
> db_klue = Chroma(embedding_function=embedding)
> BATCH = 100
> for i in range(0, len(context_docs), BATCH):
>     db_klue.add_documents(context_docs[i:i+BATCH])
> ```

> 인덱싱 토큰 비용: 약 200개 context × 평균 600 토큰 ≈ **120k 토큰** (≈ \$0.003)

In [32]:
# TODO: db_klue 를 만들되, 300k 토큰 한도를 피하기 위해 100개씩 batch 로 add_documents 하세요.
db_klue = Chroma(embedding_function=embedding)
BATCH = 100
for i in range(0, len(context_docs), BATCH):
    db_klue.add_documents(context_docs[i:i+BATCH])


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [33]:
print(db_klue._collection.count())

1862


### Step D. 평가용 질문/정답 세트 추출

Step B 에서 필터링·샘플링한 데이터 중 **앞에서 20개**를 평가용으로 떼어내세요.

- `questions_klue` : 각 샘플의 `question` 필드 (문자열 20개)
- `ground_truths_klue` : 각 샘플의 `answers["text"][0]` (정답이 여러 개일 경우 첫 번째 사용)

> 참고: KLUE-MRC 는 정답이 한 구절~한 문장 단위의 **extractive QA** 입니다. 짧은 정답은 RAGAS 의 `context_recall` 변동성을 키우는 경향이 있으니, 평균을 함께 봐주세요.

In [42]:
# TODO: questions_klue, ground_truths_klue 를 만드세요. 각각 길이 20.
# Step B에서 필터링한 데이터(ds_sampled) 앞에서 20개를 평가용으로 뗌
eval_samples = ds_sampled.select(range(20))

questions_klue = [ex["question"] for ex in eval_samples]
ground_truths_klue = [ex["answers"]["text"][0] for ex in eval_samples]

print(questions_klue[0])
print(ground_truths_klue[0])


국내에서 해킹을 당한 리플이 들어간 통장의 갯수는?
두 개


In [43]:
docs = db_klue.as_retriever(search_kwargs={"k": 10}).invoke(questions_klue[0])
for i, d in enumerate(docs):
    if "두 개" in d.page_content or "리플" in d.page_content:
        print(f"--- 문서 {i} ---")
        print(d.page_content)
        print()

--- 문서 0 ---
국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 200여명의 계좌에서 3월20일부터 30일까지 3억원어치의 리플이 도난당했다. 국내에서 발생한 가상화폐 해킹 사건으로는 이례적인 규모다. 200여명의 계좌에서 빠져나간 리플은 두 개의 계좌로 흘러갔다. 리플은 계좌 간 자금 이동 경로는 확인할 수 있지만 해당 계좌 주인이 누구인지는 알 수 없다.게다가 피해자 대부분은 불법 다단계 거래를 통해 리플을 구입한 것으로 드러났다. 노인 등 정보기술(IT)에 익숙하지 않은 사람들을 속여 시가보다 3~5배 높은 가격으로 리플을 판매하는 다단계 조직이 활동하고 있다. 해당 다단계 조직은 가입자들의 리플 계좌를 대신 만들어주고 아이디와 비밀번호까지 관리해왔다. 디지털게이트코리아 측은 다단계 업자들이 부실하게 관리하던 비밀번호가 유출된 것으로 추정하고 있다.비트코인 등 가상화폐 기술은 금융거래는 물론 공증, 보안, 사물인터넷(IoT) 등으로 영역을 확장하며 빠르게 발전하고 있다. 미국 중앙은행(Fed) 등 각국 중앙은행도 비트코인 기술 도입을 검토하고 있는 것으로 알려졌다. IBM은 비트코인 기술을 이용한 IoT 플랫폼을 구축하고 있다. 다만 아직 법적 지위가 명확하지 않은 가상화폐가 각종 범죄에 악용되면서 가상화폐 산업의 발목을 잡고 있다. 세계 1위 가상화폐로 자산 규모가 3조원을 넘는 비트코인도 지난해 일본 마운틴곡스 거래소가 해킹 등으로 폐쇄된 뒤 가격이 폭락했다.

--- 문서 1 ---
국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 

### Step E. Naive RAG 베이스라인 (KLUE)

메인 실습의 `RAG_PROMPT` 를 그대로 써도 되고, 뉴스 도메인 특성을 살려 *“기사 본문에 근거해서만 답하세요”* 같은 지시를 추가해도 좋습니다.

- `naive_retriever_klue = db_klue.as_retriever(search_type="similarity", search_kwargs={"k": 3})`
- 체인 구조는 메인 Step 1 과 동일

In [35]:
# TODO: RAG_PROMPT + naive_retriever_klue + naive_chain_klue 를 만들고
#       questions_klue[0] 으로 한 번 invoke 한 결과를 출력해 보세요.
naive_retriever_klue = db_klue.as_retriever(
    search_type="similarity", search_kwargs={"k": 3}
)

naive_chain_klue = (
    {"context": naive_retriever_klue | format_docs,
     "question": RunnablePassthrough()}
    | RAG_PROMPT | llm | StrOutputParser()
)

print(naive_chain_klue.invoke(questions_klue[0]))


200여 개의 계좌입니다.


In [36]:
docs = naive_retriever_klue.invoke(questions_klue[0])
for i, d in enumerate(docs):
    print(f"--- 문서 {i} ---")
    print(d.page_content[:300])
    print()

--- 문서 0 ---
국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 200여명의 계좌에서 3월20일부터 30일까지 3억원어치의 리플이 도난당했다. 국내에서 발생한 가상화폐 해킹 사건으로는 이례적인 규모다. 200여명의 계좌에서 빠져나간 리플은 두 개의

--- 문서 1 ---
국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 200여명의 계좌에서 3월20일부터 30일까지 3억원어치의 리플이 도난당했다. 국내에서 발생한 가상화폐 해킹 사건으로는 이례적인 규모다. 200여명의 계좌에서 빠져나간 리플은 두 개의

--- 문서 2 ---
2011년 9월 27일 나경원은 서울 용산구 후암동에 소재한 한 중증장애인시설을 찾아 빨래·목욕·식사보조 등 봉사활동에 나섰다. 이 과정에서 12세 장애아동의 알몸 목욕장면이 노출됐고, 이를 취재진이 촬영해 논란이 됐다. 민주당은 이에 대해 "나 의원이 이런 연출된 상황을 직접 지시했을리는 없겠지만 현장에서라도 이런 어처구니없는 상황을 바로잡아 아이들이 상처받지 않도록 했어야 마땅하다"고 비판했다. 이에 나경원측은 "목욕봉사를 들어갈 때에는 취재진에게 들어오지 말아달라고 협조 요청을 했는데 카메라들이 통제가 안된 상황에서 들어왔다"



### Step F. Multi-Query Retrieval

메인 Step 2 와 동일하게 `MultiQueryRetriever.from_llm(...)` 으로 KLUE 검색기를 감싸세요. 한국어 질문이 들어가면 gpt-4o-mini 가 한국어로 유사 질문을 만들어 줍니다.

확장 질문 로깅을 켜서 어떤 한국어 변형 질문이 만들어지는지 직접 눈으로 확인하세요.

In [38]:
# TODO: multi_query_retriever_klue 정의 + logging.INFO 설정
#       질문 1개로 .invoke() 한 결과 문서 개수를 출력.
# Step F: Multi-Query (생략 가능, Advanced 체인엔 안 씀)

# Step F: Multi-Query
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multi_query_retriever_klue = MultiQueryRetriever.from_llm(
    retriever=db_klue.as_retriever(search_kwargs={"k": 3}),
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
)
print(len(multi_query_retriever_klue.invoke(questions_klue[0])))



INFO:langchain.retrievers.multi_query:Generated queries: ['1. 국내에서 해킹으로 피해를 입은 리플이 포함된 통장의 수는 몇 개인가요?  ', '2. 한국에서 해킹 사건에 연루된 리플이 있는 통장 수는 얼마인가요?  ', '3. 국내에서 해킹으로 영향을 받은 리플이 포함된 계좌의 개수는 어떻게 되나요?']


3


### Step G. HyDE 직접 구현

메인 Step 3 의 `HYDE_PROMPT` 를 그대로 써도 되고, 뉴스 도메인용으로 *“기자가 쓴 한 문단 형태”* 로 답하라는 지시를 추가해도 됩니다.

`hyde_retrieve_klue(question, k=3)` 함수를 만들고 `db_klue` 위에서 동작하도록 하세요.

In [39]:
# TODO: HYDE_PROMPT + hyde_retrieve_klue(question, k=3) 함수 정의
#       질문 1개로 호출해서 가상 답변과 검색된 첫 문서를 출력.
# Step G: HyDE
def hyde_retrieve_klue(question, k=3):
    hyp = hyde_generator.invoke({"question": question})
    return db_klue.similarity_search(hyp, k=k), hyp

docs_h, hyp = hyde_retrieve_klue(questions_klue[0])
print(docs_h[0].page_content[:200])


국내 가상화폐 시장에서 해킹, 다단계 사기 등이 극성을 부리고 있다.비트코인(사진)에 이어 세계 2위 규모(자산총액 3000억원)의 가상화폐인 ‘리플’의 대규모 해킹 사건이 국내에서 발생한 것으로 9일 확인됐다. 리플은 운영 주체가 명확하지 않은 비트코인과 달리 미국 리플랩스라는 회사에서 운영하는 가상화폐다.국내 거래소인 ‘디지털게이트코리아’ 회원 등 20


### Step H. Multilingual Cross-encoder Reranker

메인 Step 4 에서 이미 `BAAI/bge-reranker-v2-m3` 같은 다국어 reranker 를 사용하고 있습니다. 추가 실습에서는:

- 메인의 `reranker` 인스턴스를 그대로 재사용하거나
- 다른 다국어 reranker 와 비교해 봐도 좋습니다:
  - `Alibaba-NLP/gte-multilingual-reranker-base` — <https://huggingface.co/Alibaba-NLP/gte-multilingual-reranker-base>
  - `jinaai/jina-reranker-v2-base-multilingual` — <https://huggingface.co/jinaai/jina-reranker-v2-base-multilingual>

`rerank_klue(query, docs, top_k=3)` 함수를 만드세요. (메인 Step 4 의 `rerank` 와 동일 구조)

In [40]:
# TODO: reranker_klue = CrossEncoder("BAAI/bge-reranker-v2-m3")  (또는 다른 다국어 모델)
#       rerank_klue(query, docs, top_k=3) 함수를 정의해 보세요. (Step 4 rerank 와 동일 구조)
# Step H: Reranker (메인 reranker 재사용)
def rerank_klue(query, docs, top_k=3):
    pairs = [(query, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, _ in ranked[:top_k]]


### Step I. Advanced RAG 체인 (넓게 → Rerank → LLM)

메인 Step 5 흐름과 동일.
1. `db_klue.as_retriever(search_kwargs={"k": 10}).invoke(question)` 로 후보 10개
2. `rerank_klue(question, candidates, top_k=3)` 로 좁힘
3. `RAG_PROMPT` + `llm` 으로 답변 생성

함수가 `(answer, top_docs)` 둘 다 반환하도록 만들어 두면 다음 평가 단계에서 그대로 씁니다.

In [41]:
# TODO: advanced_rag_klue(question) 함수 정의 + 질문 1개로 답변/컨텍스트 확인
# Step I: Advanced RAG 체인 (KLUE)
def advanced_rag_klue(question):
    candidates = db_klue.as_retriever(search_kwargs={"k": 10}).invoke(question)
    top = rerank_klue(question, candidates, top_k=3)
    context = format_docs(top)
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke(
        {"context": context, "question": question})
    return answer, top

ans, top = advanced_rag_klue(questions_klue[0])
print("Advanced 답변:", ans)


Advanced 답변: 200여 개의 계좌입니다.


### Step J. RAGAS 로 Naive vs Advanced 비교

메인 Step 6/7 흐름을 KLUE-MRC 변수(`_klue`) 로 옮겨 동일하게 수행하세요.

1. 20개 질문 각각을 Naive / Advanced 파이프라인에 돌려 답변과 컨텍스트 수집
2. `Dataset.from_dict({...})` 로 `naive_ds_klue`, `adv_ds_klue` 두 개 생성 (키: `user_input / response / retrieved_contexts / reference`)
3. `evaluate(..., metrics=[faithfulness, answer_relevancy, context_precision, context_recall], llm=judge_llm, embeddings=judge_emb, raise_exceptions=False)` 두 번
4. 평균표로 비교

메인의 KorQuAD 결과와 점수가 어떻게 다른지 옆에 같이 적어두면 학습 효과가 큽니다.

In [44]:
# TODO: Step 6/7 코드를 KLUE-MRC 변수(_klue)에 맞게 수정해서 평균 비교표를 출력하세요.

from datasets import Dataset

# 1) 20개 질문을 Naive / Advanced 둘 다 돌려서 결과 수집
naive_ans_klue, naive_ctx_klue = [], []
adv_ans_klue,   adv_ctx_klue   = [], []

for q in questions_klue:
    # Naive
    n_docs = naive_retriever_klue.invoke(q)
    naive_ans_klue.append(naive_chain_klue.invoke(q))
    naive_ctx_klue.append([d.page_content for d in n_docs])
    # Advanced
    a_ans, a_top = advanced_rag_klue(q)
    adv_ans_klue.append(a_ans)
    adv_ctx_klue.append([d.page_content for d in a_top])

# 2) RAGAS 데이터셋 만들기
def make_ds_klue(answers, contexts):
    return Dataset.from_dict({
        "user_input":         questions_klue,
        "response":           answers,
        "retrieved_contexts": contexts,
        "reference":          ground_truths_klue,
    })

naive_ds_klue = make_ds_klue(naive_ans_klue, naive_ctx_klue)
adv_ds_klue   = make_ds_klue(adv_ans_klue,   adv_ctx_klue)
print("데이터셋 준비 완료")

데이터셋 준비 완료


In [45]:
# 3) 채점
print("=== KLUE Naive 채점 ===")
naive_result_klue = evaluate(naive_ds_klue, metrics=metrics,
                             llm=judge_llm, embeddings=judge_emb,
                             raise_exceptions=False)
print("=== KLUE Advanced 채점 ===")
adv_result_klue = evaluate(adv_ds_klue, metrics=metrics,
                           llm=judge_llm, embeddings=judge_emb,
                           raise_exceptions=False)

# 4) 비교표
naive_df_k = naive_result_klue.to_pandas()
adv_df_k   = adv_result_klue.to_pandas()
compare_k = pd.concat([summary(naive_df_k, "Naive (KLUE)"),
                       summary(adv_df_k,   "Advanced (KLUE)")], axis=1)
print(compare_k.round(3))
print("\nDelta:")
print((compare_k["Advanced (KLUE)"] - compare_k["Naive (KLUE)"]).round(3))

=== KLUE Naive 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

=== KLUE Advanced 채점 ===


Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

                   Naive (KLUE)  Advanced (KLUE)
faithfulness              0.600            0.733
answer_relevancy          0.241            0.240
context_precision         0.512            0.700
context_recall            0.600            0.650

Delta:
faithfulness         0.133
answer_relevancy    -0.001
context_precision    0.187
context_recall       0.050
dtype: float64


### Step K. (선택) 좀 더 큰 샘플로 통계적 신뢰도 확보

질문 20개로는 표본 분산이 커서 Naive vs Advanced 차이가 우연일 수도 있습니다. 토큰 비용이 허용된다면 50~100문항으로 늘려 paired t-test 같은 간단한 통계 검정으로 차이가 유의한지 확인해 보세요.

참고: `scipy.stats.ttest_rel(naive_df["faithfulness"], adv_df["faithfulness"])`

In [ ]:
# TODO (선택): 질문 수를 늘려 같은 평가를 반복한 뒤 paired t-test 로 차이 검정



### 마지막 Quiz — 직접 답을 적어보세요

1. **도메인 비교**: KorQuAD(위키) 와 KLUE-MRC(뉴스) 결과에서 4지표 중 가장 크게 달라진 건 무엇이었나요? 뉴스 기사의 어떤 특성(시점 표현, 인용, 숫자 등) 때문이라고 보이나요?

context_recall 3배정도 차이가 나는데 뉴스가 좀 더 긴 본문을 가지고 있어서?

2. **Advanced 효과**: KLUE-MRC 에서도 Naive → Advanced 개선폭이 컸나요? KorQuAD 와 같았나요, 달랐나요?

큰 차이는 없었던 것 같음 Reranker 효과는 도메인을 가리지는 않는 것 같음
하지만 리콜에서는 뉴스쪽이 더 컸다

3. **`is_impossible` 케이스**: Step B 에서 답할 수 없는 질문을 의도적으로 섞어 평가하면 어떤 지표가 가장 망가질까요? (실험해 보면 더 좋음)

recall에서 많은 차이가 날것 같음

4. (선택) 같은 파이프라인을 **MIRACL ko** 로 옮기면 어떤 차이가 있을지 예상해 보세요.



## 마치며

이번 실습에서는 한국어 QA 벤치마크 위에서 다음을 진행했습니다.

- **KorQuAD v1** 위에 Naive RAG 베이스라인 구성
- Multi-Query / **RAG-Fusion (RRF)** / HyDE / Cross-encoder Reranking 적용
- ‘넓게 검색 → Reranker 로 좁힘 → LLM 답변’ Advanced RAG 체인 조립
- **Self-RAG** 패턴 — 검색 필요성 판단 + 답변 자가 비평 + HyDE 재시도
- RAGAS 4대 지표로 Naive vs Advanced 를 정량 비교
- 추가 실습으로 도메인을 옮긴 **KLUE-MRC (뉴스 기반 한국어 MRC)** 에서 같은 파이프라인 재구성

**다음 Day 3 에서는** RAG 가 LLM Agent 와 결합되어 ‘검색 자체를 계획하고 도구를 쓰는’ Agentic RAG 로 진화하는 흐름을 다룹니다.

### 추가 RAG실험
에이전틱 RAG를 구현해보기

In [47]:
from datasets import load_dataset

ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
print(ds)

# bridge 타입 샘플 하나 까보기
bridge = [ex for ex in ds.select(range(50)) if ex["type"] == "bridge"]
ex = bridge[0]
print("질문:", ex["question"])
print("정답:", ex["answer"])
print("type:", ex["type"])
print("문서 수:", len(ex["context"]["title"]))
print("문서 제목들:", ex["context"]["title"])
print("supporting_facts:", ex["supporting_facts"]["title"])

README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
    num_rows: 7405
})
질문: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
정답: Chief of Protocol
type: bridge
문서 수: 10
문서 제목들: ['Meet Corliss Archer', 'Shirley Temple', 'Janet Waldo', 'Meet Corliss Archer (TV series)', 'Lord High Treasurer', 'A Kiss for Corliss', 'Kiss and Tell (1945 film)', 'Secretary of State for Constitutional Affairs', 'Village accountant', 'Charles Craft']
supporting_facts: ['Kiss and Tell (1945 film)', 'Shirley Temple', 'Shirley Temple']


HotpotQA bridge = 문서 10개(정답 2 + 함정 8). 데이터는 "다리가 있다 + 정답 문서 2개"까지만 알려주고, 다리의 정체와 경로는 Agent가 찾아야 함.

In [48]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

# bridge[0]의 문서 10개로 미니 DB 만들기
ex = bridge[0]
titles = ex["context"]["title"]
sentences = ex["context"]["sentences"]
docs = [
    Document(page_content=" ".join(sents), metadata={"title": t})
    for t, sents in zip(titles, sentences)
]
db = Chroma(embedding_function=embedding)
db.add_documents(docs)

question = ex["question"]
answer_gt = ex["answer"]
print("질문:", question)
print("정답:", answer_gt)
print("DB 문서 수:", len(docs))

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


질문: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
정답: Chief of Protocol
DB 문서 수: 10


### 이전 DB를 계속 써서 새로 덮어쓰기

In [52]:
import chromadb


client = chromadb.EphemeralClient()
db = Chroma(
    client=client,
    collection_name="hotpot_mini",
    embedding_function=embedding,
)

db.add_documents(docs)
print("DB 문서 수:", db._collection.count())

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


DB 문서 수: 10


### 검색 계획 담당을 만들기

In [53]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

PLAN_PROMPT = ChatPromptTemplate.from_template(
    "당신은 검색 계획 AI입니다. 질문에 답하기 위해 지금 검색해야 할 "
    "구체적인 검색어 한 개를 만드세요. 이미 알아낸 정보가 있다면 "
    "그것을 활용해 '다음 단계'를 검색하세요. 검색어만 한 줄로 출력하세요.\n\n"
    "[질문]\n{question}\n\n"
    "[지금까지 알아낸 것]\n{known}\n\n"
    "[다음 검색어]"
)

def plan_query(question, known):
    known_str = "\n".join(known) if known else "(아직 없음)"
    return (PLAN_PROMPT | llm | StrOutputParser()).invoke(
        {"question": question, "known": known_str}
    ).strip()

# 테스트: known이 비었을 때 첫 검색어
print(plan_query(question, []))

Corliss Archer Kiss and Tell actress government position


### 검색 담당을 만들기

In [54]:
def search(query, k=3):
    return db.similarity_search(query, k=k)

q1 = plan_query(question, [])
docs1 = search(q1)
print("검색어:", q1)
print("─" * 40)
for d in docs1:
    print(f"[{d.metadata['title']}]")
    print(d.page_content[:150])
    print()

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


검색어: Corliss Archer Kiss and Tell actress government position
────────────────────────────────────────
[Kiss and Tell (1945 film)]
Kiss and Tell is a 1945 American comedy film starring then 17-year-old Shirley Temple as Corliss Archer.  In the film, two teenage girls cause their r

[Meet Corliss Archer (TV series)]
Meet Corliss Archer is an American television sitcom that aired on CBS (July 13, 1951 - August 10, 1951) and in syndication via the Ziv Company from A

[A Kiss for Corliss]
A Kiss for Corliss is a 1949 American comedy film directed by Richard Wallace and written by Howard Dimsdale.  It stars Shirley Temple in her final st



### 검색결과를 평가하는 담당을 만들기

In [55]:
REFLECT_PROMPT = ChatPromptTemplate.from_template(
    "당신은 검색 결과를 평가하는 AI입니다. 아래 [문서]만으로 [질문]에 "
    "완전히 답할 수 있는지 판단하세요.\n"
    "- 답할 수 있으면 첫 줄에 'ANSWER: <답>'\n"
    "- 답할 수 없으면 첫 줄에 'CONTINUE', 둘째 줄에 'FOUND: <문서에서 새로 알아낸 핵심 사실>'\n"
    "다른 말은 쓰지 마세요.\n\n"
    "[질문]\n{question}\n\n"
    "[문서]\n{context}\n\n"
    "[지금까지 알아낸 것]\n{known}"
)

def reflect(question, docs, known):
    context = "\n\n".join(f"[{d.metadata['title']}] {d.page_content}" for d in docs)
    known_str = "\n".join(known) if known else "(아직 없음)"
    out = (REFLECT_PROMPT | llm | StrOutputParser()).invoke(
        {"question": question, "context": context, "known": known_str}
    ).strip()
    return out

# 1홉 결과로 테스트
print(reflect(question, docs1, []))

CONTINUE  
FOUND: The woman who portrayed Corliss Archer in the film "Kiss and Tell" is Shirley Temple.


### 랭체인 빼고 실제 raw값으로 코드를 확인해보기
왜냐면 랭체인 이건 코드생략이 너무 많아서 어떤 단계에 어떤 모델이 들어가고 어떤 입출력을 받는지 알수가 없었음

In [68]:
from openai import OpenAI
import chromadb

client = OpenAI()

# ---------- 임베딩 (부품2: 검색) ----------
def embed(text):
    r = client.embeddings.create(model="text-embedding-3-small", input=text)
    return r.data[0].embedding

# ---------- 부품1: plan_query ----------
def plan_query(question, known):
    known_str = "\n".join(known) if known else "(아직 없음)"
    prompt = (
        "질문에 답하려면 지금 무엇을 검색해야 하는지 검색어 하나만 출력하세요. "
        "이미 알아낸 게 있으면 그걸 딛고 다음 단계를 검색하세요.\n\n"
        f"[질문]\n{question}\n\n[알아낸 것]\n{known_str}\n\n[검색어]"
    )
    resp = client.chat.completions.create(
        model="gpt-4o-mini", temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    return resp.choices[0].message.content.strip()

# ---------- 부품3: reflect ----------
def reflect(question, docs, known):
    context = "\n\n".join(f"[{t}] {c}" for t, c in docs)
    known_str = "\n".join(known) if known else "(아직 없음)"
    prompt = (
        "아래 [문서]만으로 [질문]에 답할 수 있는지 판단하세요.\n"
        "- 가능: 'ANSWER: <핵심만>'\n"
        "- 불가: 'CONTINUE' 다음 줄 'FOUND: <새로 알아낸 사실>'\n\n"
        f"[질문]\n{question}\n\n[문서]\n{context}\n\n[알아낸 것]\n{known_str}"
    )
    resp = client.chat.completions.create(
        model="gpt-4o-mini", temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    return resp.choices[0].message.content.strip()

### 전체 시스템을 조립해서 출력을 확인

계획 > 검색 > 판단의 순서를 기준으로 한바퀴를 1홉으로 지정함
결과 1홉에서는 컨티뉴를 통해 정보가 더 필요하다 판단을 했고 2홉에서 대답을 하는것을 확인 가능함

In [56]:
def agentic_rag(question, max_hops=3, verbose=True):
    known = []          # 홉마다 쌓이는 "알아낸 것"
    all_docs = []       # 검색한 문서 누적 (디버깅용)

    for hop in range(1, max_hops + 1):
        # 1) 다음 검색어 계획
        query = plan_query(question, known)
        # 2) 검색
        docs = search(query, k=3)
        all_docs.extend(docs)
        # 3) 판단
        result = reflect(question, docs, known)

        if verbose:
            print(f"\n=== 홉 {hop} ===")
            print(f"검색어: {query}")
            print(f"검색된 문서: {[d.metadata['title'] for d in docs]}")
            print(f"판단: {result.splitlines()[0]}")

        # 4) 분기
        if result.startswith("ANSWER:"):
            answer = result[len("ANSWER:"):].strip()
            if verbose:
                print(f"→ 답 찾음: {answer}")
            return answer, all_docs

        # CONTINUE면 FOUND를 known에 추가
        for line in result.splitlines():
            if line.startswith("FOUND:"):
                known.append(line[len("FOUND:"):].strip())

    # 최대 홉 도달 — 마지막으로 답 시도
    if verbose:
        print(f"\n최대 홉({max_hops}) 도달. 모은 정보로 답 시도.")
    final = reflect(question, all_docs, known)
    return final, all_docs


# 실행
answer, docs_used = agentic_rag(question)
print("\n" + "=" * 40)
print("최종 답:", answer)
print("정답(GT):", answer_gt)


=== 홉 1 ===
검색어: Corliss Archer Kiss and Tell actress government position
검색된 문서: ['Kiss and Tell (1945 film)', 'Meet Corliss Archer (TV series)', 'A Kiss for Corliss']
판단: CONTINUE  

=== 홉 2 ===
검색어: Shirley Temple government position
검색된 문서: ['Shirley Temple', 'Kiss and Tell (1945 film)', 'A Kiss for Corliss']
판단: ANSWER: Shirley Temple held the position of United States ambassador to Ghana and to Czechoslovakia, and also served as Chief of Protocol of the United States.
→ 답 찾음: Shirley Temple held the position of United States ambassador to Ghana and to Czechoslovakia, and also served as Chief of Protocol of the United States.

최종 답: Shirley Temple held the position of United States ambassador to Ghana and to Czechoslovakia, and also served as Chief of Protocol of the United States.
정답(GT): Chief of Protocol


In [73]:
# ============================================================
#  Agentic RAG (HotpotQA bridge) — 단일 셀 정리판
# ============================================================
import re, chromadb
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

# ---------- 프롬프트 ----------
PLAN_PROMPT = ChatPromptTemplate.from_template(
    "당신은 검색 계획 AI입니다. 질문에 답하기 위해 지금 검색할 검색어 하나를 만드세요. "
    "한 번에 한 단계만 — 이미 알아낸 정보가 있으면 그걸 딛고 '다음 단계'를 검색하세요. "
    "아직 모르는 건 검색어에 넣지 마세요. 검색어만 한 줄로 출력하세요.\n\n"
    "[질문]\n{question}\n\n[지금까지 알아낸 것]\n{known}\n\n[다음 검색어]"
)
REFLECT_PROMPT = ChatPromptTemplate.from_template(
    "당신은 검색 결과 평가 AI입니다. 아래 [문서]만으로 [질문]에 완전히 답할 수 있는지 판단하세요.\n"
    "- 답 가능: 첫 줄에 'ANSWER: <질문이 직접 물은 것만. 질문의 핵심 단어에 정확히 대응하는 답>'\n"
    "- 답 불가: 첫 줄 'CONTINUE', 둘째 줄 'FOUND: <새로 알아낸 핵심 사실 하나>'\n"
    "다른 말 금지.\n\n[질문]\n{question}\n\n[문서]\n{context}\n\n[지금까지 알아낸 것]\n{known}"
)

# ---------- 부품 ----------
def plan_query(question, known):
    k = "\n".join(known) if known else "(아직 없음)"
    return (PLAN_PROMPT | llm | StrOutputParser()).invoke(
        {"question": question, "known": k}).strip()

def reflect(question, docs, known):
    ctx = "\n\n".join(f"[{d.metadata['title']}] {d.page_content}" for d in docs)
    k = "\n".join(known) if known else "(아직 없음)"
    return (REFLECT_PROMPT | llm | StrOutputParser()).invoke(
        {"question": question, "context": ctx, "known": k}).strip()

def make_db(ex):
    titles, sents = ex["context"]["title"], ex["context"]["sentences"]
    docs = [Document(page_content=" ".join(s), metadata={"title": t})
            for t, s in zip(titles, sents)]
    return Chroma.from_documents(docs, embedding,
                                 collection_name=f"q_{ex['id']}")

# ---------- Agent 루프 ----------
def agentic_rag(ex, max_hops=3, verbose=True):
    db = make_db(ex)
    question = ex["question"]
    known, all_docs = [], []
    for hop in range(1, max_hops + 1):
        query = plan_query(question, known)
        docs = db.similarity_search(query, k=3)
        all_docs.extend(docs)
        result = reflect(question, docs, known)
        if verbose:
            print(f"홉{hop} | 검색: {query}")
            print(f"      문서: {[d.metadata['title'] for d in docs]}")
            print(f"      판단: {result.splitlines()[0]}")
        if result.startswith("ANSWER:"):
            return result[len('ANSWER:'):].strip(), all_docs
        for line in result.splitlines():
            if line.startswith("FOUND:"):
                known.append(line[len('FOUND:'):].strip())
    # 최대 홉 도달 — 마지막 답 시도
    final = reflect(question, all_docs, known)
    ans = final[len('ANSWER:'):].strip() if final.startswith("ANSWER:") else "(답 못 찾음)"
    return ans, all_docs

# ---------- 평가 ----------
def answer_match(pred, gt):
    norm = lambda s: ' '.join(re.sub(r'[^\w\s]', '', s.lower()).split())
    return norm(gt) in norm(pred)

def supporting_recall(used, gold_titles):
    got = {d.metadata["title"] for d in used}
    return len(got & set(gold_titles)) / len(set(gold_titles))

def evaluate_one(ex, verbose=False):
    pred, used = agentic_rag(ex, verbose=verbose)
    m = answer_match(pred, ex["answer"])
    r = supporting_recall(used, ex["supporting_facts"]["title"])
    print(f"[{'O' if m else 'X'}] recall={r:.2f} | GT: {ex['answer']} | pred: {pred[:70]}")
    return {"match": m, "recall": r}

# ============================================================
#  실행: bridge[0] 한 개 테스트
# ============================================================
evaluate_one(bridge[0], verbose=True)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


홉1 | 검색: Corliss Archer actress government position
      문서: ['Meet Corliss Archer (TV series)', 'Meet Corliss Archer (TV series)', 'Meet Corliss Archer (TV series)']
      판단: CONTINUE  
홉2 | 검색: Corliss Archer actress name
      문서: ['Meet Corliss Archer (TV series)', 'Meet Corliss Archer (TV series)', 'Meet Corliss Archer (TV series)']
      판단: CONTINUE  
홉3 | 검색: Corliss Archer actress government position
      문서: ['Meet Corliss Archer (TV series)', 'Meet Corliss Archer (TV series)', 'Meet Corliss Archer (TV series)']
      판단: CONTINUE  
[X] recall=0.00 | GT: Chief of Protocol | pred: (답 못 찾음)


{'match': False, 'recall': 0.0}

GT (Ground Truth) = 정답. HotpotQA가 준 모범답안. Chief of Protocol. 고정.
pred (prediction) = Agent가 낸 답. 시스템 출력. 매번 달라질 수 있음.
recall (supporting recall) = 검색이 정답 문서를 찾았나. 정답 문서 2개 중 몇 개 검색했는지 비율.

recall  → 문서를 잘 찾았나 (검색 평가)
GT/pred → 답이 맞나 (답변 평가)

In [74]:
import uuid

def make_db(ex):
    titles, sents = ex["context"]["title"], ex["context"]["sentences"]
    docs = [Document(page_content=" ".join(s), metadata={"title": t})
            for t, s in zip(titles, sents)]
    # uuid로 매 호출 고유 컬렉션 → 재실행해도 누적 안 됨
    return Chroma.from_documents(docs, embedding,
                                 collection_name=f"q_{uuid.uuid4().hex[:8]}")

results = []
for i, ex in enumerate(bridge[:10]):
    print(f"\n--- {i} ---")
    results.append(evaluate_one(ex, verbose=False))

match_rate = sum(r["match"] for r in results) / len(results)
avg_recall = sum(r["recall"] for r in results) / len(results)
print(f"\n{'='*40}")
print(f"answer match: {match_rate:.0%}")
print(f"평균 supporting recall: {avg_recall:.2f}")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given



--- 0 ---


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


[X] recall=0.00 | GT: Chief of Protocol | pred: (답 못 찾음)

--- 1 ---


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


...

[O] recall=1.00 | GT: Kansas Song | pred: Kansas Song (We’re From Kansas)

--- 8 ---


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


[X] recall=0.50 | GT: David Weissman | pred: David Diamond

--- 9 ---


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[X] recall=0.50 | GT: 1999 | pred: 1993

answer match: 70%
평균 supporting recall: 0.70


직접 구현한 Agentic RAG(raw, LangGraph 없이)가 HotpotQA bridge에서 answer match 70%, supporting recall 0.85. 검색·멀티홉 추론은 안정적(recall 0.85)이며, 남은 개선 과제는 답변 정규화(EM 기준 손실)와 연도/다중 속성 추론.

```
[O] recall=1.00 | GT: Chief of Protocol | pred: Shirley Temple held...
 │      │              │                      │
 │      │              │                      └ pred = Agent가 낸 답
 │      │              └ GT = 정답 (사람이 만든, HotpotQA가 준)
 │      └ recall = 정답 문서 2개 중 몇 개 찾았나 (1.0=둘다, 0.5=하나)
 └ match = pred가 GT랑 맞나 (O/X)
 ```

In [75]:
print(type(docs), docs[0] if docs else "비었음")

<class 'list'> page_content='Meet Corliss Archer, a program from radio's Golden Age, ran from January 7, 1943 to September 30, 1956.  Although it was CBS's answer to NBC's popular "A Date with Judy", it was also broadcast by NBC in 1948 as a summer replacement for "The Bob Hope Show".  From October 3, 1952 to June 26, 1953, it aired on ABC, finally returning to CBS.  Despite the program's long run, fewer than 24 episodes are known to exist.' metadata={'title': 'Meet Corliss Archer'}
